In [ ]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
import ollama

BASE_DIR = Path(r"E:/Data_for_Practice/JapMedia")  # edit this path
ARTICLE_DIR = BASE_DIR / "data" / "embed_figure"
LABEL_DIRS = [BASE_DIR / "data" / "kor_data", BASE_DIR / "data" / "jap_data" / "translated"]
OUTPUT_DIR = BASE_DIR / "validation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_NAME = "llama4"  # use and record the exact locally installed model tag/digest
RANDOM_SEED = 42
TOP_K, RANDOM_K = 3, 2
MAX_CHARS_PER_ARTICLE = 3000
ARTICLE_COLUMNS = ["본문_en", "영문 번역", "본문", "article_text", "text"]
article_files = sorted(ARTICLE_DIR.glob("*_doc_topics_with_emb.csv"))
if not article_files:
    raise FileNotFoundError(f"No article-level topic CSVs in {ARTICLE_DIR}")
print(f"Article files: {len(article_files)}")

In [ ]:
def make_items(path, rng):
    df = pd.read_csv(path, encoding="utf-8-sig", low_memory=False)
    body_col = next((c for c in ARTICLE_COLUMNS if c in df.columns), None)
    if not body_col:
        raise ValueError(f"{path.name}: original article body missing; available columns: {list(df.columns)[:30]}")
    for c in ["topic", "topic_label", "newspaper", "year"]:
        if c not in df.columns: raise ValueError(f"{path.name}: missing {c}")
    emb_cols = sorted([c for c in df.columns if re.fullmatch(r"emb_\d+", c)], key=lambda c: int(c[4:]))
    if not emb_cols: raise ValueError(f"{path.name}: no embedding columns")
    df["topic"] = pd.to_numeric(df["topic"], errors="coerce")
    df = df.loc[df["topic"].notna() & (df["topic"] != -1)].copy()
    items = []
    for topic, group in df.groupby("topic", sort=True):
        group = group.copy()
        x = group[emb_cols].to_numpy(dtype=float)
        centroid = x.mean(axis=0)
        den = np.linalg.norm(x, axis=1) * np.linalg.norm(centroid)
        group["centroid_similarity"] = np.divide(x @ centroid, den, out=np.full(len(group), np.nan), where=den > 0)
        group["original_row"] = group.index
        top_idx = group.sort_values(["centroid_similarity", "original_row"], ascending=[False, True], na_position="last").head(TOP_K).index.tolist()
        rest = group.drop(index=top_idx)
        random_idx = rng.choice(rest.index.to_numpy(), size=min(RANDOM_K,len(rest)), replace=False).tolist()
        articles = []
        for idx, selection in [(i,"centroid") for i in top_idx] + [(i,"random") for i in random_idx]:
            row = group.loc[idx]
            raw = "" if pd.isna(row[body_col]) else str(row[body_col])
            articles.append({"original_row":int(idx), "selection":selection,
                             "centroid_similarity":float(row.centroid_similarity),
                             "text":raw[:MAX_CHARS_PER_ARTICLE], "original_chars":len(raw),
                             "truncated":len(raw)>MAX_CHARS_PER_ARTICLE})
        items.append({"source_file":path.name, "newspaper":str(group.newspaper.iloc[0]),
                      "year":int(group.year.iloc[0]), "topic":int(topic),
                      "n_articles":len(group), "body_column":body_col,
                      "topic_keywords":str(group.topic_label.iloc[0]), "articles":articles})
    return items

rng = np.random.default_rng(RANDOM_SEED)
items = [item for path in article_files for item in make_items(path,rng)]
print(f"Retained topics: {len(items)}; article excerpts: {sum(len(x['articles']) for x in items)}")
print(f"Truncated article texts: {sum(a['truncated'] for x in items for a in x['articles'])}")
with open(OUTPUT_DIR / "frame_review_inputs.json", "w", encoding="utf-8") as f:
    json.dump(items,f,ensure_ascii=False,indent=2)